# 1. 下準備

## 1-1. インポート

In [1]:
!pip install riotwatcher
!pip install pandas


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import time
import json
import os
from riotwatcher import LolWatcher, ApiError, RiotWatcher
from datetime import datetime
from pandas import json_normalize
import pandas as pd
import sqlite3

## 1-2. 初期設定

In [ ]:
api_key = 'APIkeyを入力' # Riot Developer Portalで取得したAPIキーを入力
watcher = LolWatcher(api_key)
routing = 'asia'    # 試合データ取得のデータセンター設定
region = 'jp1'      # 試合データ取得のサーバー設定

# 2. データ取得

## 2.1 試合データの取得とjsonファイルへの保存

### 2-1-1. 関数：試合データ取得とjsonファイル保存の定義

In [ ]:
def collect_rank_data(tier, division, matches_per_player=25): # matches_per_playerはプレイヤーごとに取得する試合数を指定
    all_match_data = []
    collected_match_ids = set() # 重複しない試合データのためのセット
    get_count = 0   # 取得したプレイヤー数カウンタ

    try:
        print(f"--- {tier} {division} の収集開始 ---")
        page_cnt = 1    # プレイヤーリストのページカウンタ

        # 試合数が2500に達するまでループ
        while len(all_match_data) < 2500:
            # 安全策として20ページで打ち切り
            if page_cnt > 20:
                break
            # プレイヤーリスト取得
            players = watcher.league.entries(region, 'RANKED_SOLO_5x5', tier, division, page_cnt)
            print(players[0])

            # プレイヤーリストから試合データを収集
            for player_info in players:
                try:
                    if len(all_match_data) >=2500: # 試合数が2500に達したら終了
                        break
                    if player_info['wins'] + player_info['losses'] < 30: # プレイ試合が30試合未満のプレイヤーはスキップ
                        continue

                    # 試合IDリストの取得 (データセンター, puuid, 取得試合数, キュータイプ: 420（ソロランク）)
                    match_ids = watcher.match.matchlist_by_puuid(routing, player_info['puuid'], count=matches_per_player, queue=420)

                    # 試合IDから試合のデータを取得
                    for m_id in match_ids:
                        if m_id not in collected_match_ids: # 一度取得した試合はスキップ
                            m_detail = watcher.match.by_id(routing, m_id)   # 試合データの取得
                            all_match_data.append(m_detail) # 試合データをリストに追加
                            collected_match_ids.add(m_id)   # 取得済み試合IDセットに追加

                    get_count += 1  

                    if(get_count % 10 == 0):
                        print(f"--- 現在 {get_count} 人のプレイヤーからデータ取得中 ---")
                    
                except ApiError as err:
                    print(f"プレイヤーの取得でスキップが発生しました: {err}")
                    continue
            page_cnt += 1

        # JSONファイルに書き出し
        file_name = f"{tier}_{division}_matches.json"
        with open(file_name, 'w', encoding='utf-8') as f:
            json.dump(all_match_data, f, ensure_ascii=False, indent=4)
        
        print(f"--- {tier} {division} 完了！ 合計 {len(all_match_data)} 試合保存しました ---")

    except ApiError as err:
        print(f"致命的なエラーが発生しました: {err}")

In [ ]:
# 2500試合取得するのに1時間程度かかるため、十分な時間を確保してから実行してください。一つのランク帯のデータを取得するには、4つのディビジョン（IV, III, II, I）で合計10000試合となります。
# ランク・ティアごとにデータ収集を実行
# ランク・ティアのリスト初期化
Get_tier_list = ['IRON', 'BRONZE', 'SILVER', 'GOLD', 'PLATINUM', 'EMERALD', 'DIAMOND']
Get_division_list = ['IV', 'III', 'II', 'I']

for tier in Get_tier_list:
    for division in Get_division_list:
        collect_rank_data(tier, division)

### 2-1-2. 出力ファイル確認

In [ ]:
# debug：出力ファイル確認
# 読み込みたいファイル名
file_name = 'IRON_IV_matches.json'

# ファイルを開いて Pythonのリストや辞書に変換する
with open(file_name, 'r', encoding='utf-8') as f:
    data = json.load(f)

# 中身の確認
print(f"合計 {len(data)} 件の試合データを読み込みました。")
print(data[0])  # 最初の1試合分を表示

## 2-3. Master以上のデータ取得

### 2-3-1. Master以上の人数の確認

In [3]:
# リーグ情報を取得
league_data_m = watcher.league.masters_by_queue(region, 'RANKED_SOLO_5x5')
league_data_gm = watcher.league.grandmaster_by_queue(region, 'RANKED_SOLO_5x5')
league_data_c = watcher.league.challenger_by_queue(region, 'RANKED_SOLO_5x5')
entries_m = league_data_m['entries']
entries_gm = league_data_gm['entries']
entries_c = league_data_c['entries']

print(len(entries_m))
print(len(entries_gm))
print(len(entries_c))

1866
100
50


### 2-3-2. 関数：Master以上の試合データ取得

In [ ]:
# 三つのランク帯での取得で7500試合取得するのに1時間程度かかるため、十分な時間を確保してから実行してください
DB_NAME = 'lol_data.db'
TIERS = ['master', 'grandmaster', 'challenger']
MATCH_COUNT = 100  # 各プレイヤーから取得する試合数
TARGET_TOTAL_MATCHES = 2500  # 目標の合計試合数

# データベースに試合データを保存する関数
def save_match_to_db(cursor, match_id, tier, match_data):
    """取得したデータを指定のテーブル構造で保存"""
    sql = """
    INSERT OR IGNORE INTO matches 
    (match_id, tier, division, data, average_tier, average_division) 
    VALUES (?, ?, ?, ?, ?, ?)
    """
    # average_tier, average_division は空(None)で設定
    cursor.execute(sql, (
        match_id,
        tier.upper(),
        'I',                    # Master以上は通常 Division I 扱い
        json.dumps(match_data), # JSONを文字列化
        None,                   # 平均ティア（空）
        None                    # 平均ディビジョン（空）
    ))

# メイン処理
def main():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    for tier in TIERS:
        print(f"--- Processing {tier} ---")
        # MASTER以上のデータ数をカウント
        cursor.execute(f"SELECT COUNT(*) FROM matches WHERE tier = '{tier.upper()}'")
        collected_count = cursor.fetchone()[0]
        print(f"{tier.upper()}の現在データ数: {collected_count}件")
        
        try:
            # プレイヤーリスト取得
            if tier == 'master':
                league_data = watcher.league.masters_by_queue(region, 'RANKED_SOLO_5x5')
            elif tier == 'grandmaster':
                league_data = watcher.league.grandmaster_by_queue(region, 'RANKED_SOLO_5x5')
            elif tier == 'challenger':
                league_data = watcher.league.challenger_by_queue(region, 'RANKED_SOLO_5x5')

            entries = league_data['entries']
            
            # プレイヤーリストから試合データを収集
            for entry in entries: # リストの先頭から一人ずつ見ていく
                if collected_count > TARGET_TOTAL_MATCHES:
                    break
                
                # PUUID取得
                try:
                    puuid = entry.get('puuid')
                except ApiError as e:
                    print(f"Summoner Error: {e}")
                    continue
                
                # Match IDリスト取得
                match_ids = watcher.match.matchlist_by_puuid(routing, puuid, count=MATCH_COUNT, queue=420)
                
                for m_id in match_ids:
                    if collected_count > TARGET_TOTAL_MATCHES:
                        break
                    # 重複チェック
                    cursor.execute("SELECT 1 FROM matches WHERE match_id = ?", (m_id,))
                    if cursor.fetchone():
                        continue

                    # 試合詳細取得
                    try:
                        match_detail = watcher.match.by_id(routing, m_id)
                        # DB保存
                        save_match_to_db(cursor, m_id, tier, match_detail)
                        collected_count += 1
                        if collected_count % 100 == 0:
                            print(f"Saved: {m_id} ({tier})")
                            conn.commit()
                    except ApiError as e:
                        print(f"Match Detail Error {m_id}: {e}")
                    
                    time.sleep(1.2) # API制限回避（Development Key用）

        except ApiError as err:
            print(f"API Error in {tier}: {err}")
            if err.response.status_code == 429:
                print("Rate limit exceeded. Waiting 30s...")
                time.sleep(30)

    conn.commit()
    conn.close()
    print("Process Complete.")

if __name__ == "__main__":
    main()

--- Processing master ---
MASTERの現在データ数: 2501件
--- Processing grandmaster ---
GRANDMASTERの現在データ数: 2501件
--- Processing challenger ---
CHALLENGERの現在データ数: 1407件
API Error in challenger: 400 Client Error: Bad Request for url: https://asia.api.riotgames.com/lol/match/v5/matches/by-puuid/dNtI6Onx_vFKOCIpSbRO1g1diRoOPqhJMK0T5xudPagxreMqrTek-rplnrOnPbRmIIpwY3C6YXScQw/ids?count=500&queue=420
Process Complete.


### 2-3-3. 関数：ランク帯ごとに試合データが何試合あるのか取得する。

In [ ]:
# debug：DBの保存状況確認
def check_db_stats():
    conn = sqlite3.connect('lol_data.db')
    
    # ランクごとの件数を集計
    query = """
    SELECT 
        tier, 
        COUNT(*) as match_count,
        MAX(match_id) as latest_match_id
    FROM matches 
    GROUP BY tier
    """
    df = pd.read_sql(query, conn)
    conn.close()
    
    if df.empty:
        print("DBは空です。まだデータが保存されていません。")
    else:
        print("--- 現在のDB保存状況 ---")
        print(df)

check_db_stats()

--- 現在のDB保存状況 ---
          tier  match_count latest_match_id
0       BRONZE         9609   JP1_563246635
1   CHALLENGER          197   JP1_570859978
2      DIAMOND         8923   JP1_563543148
3      EMERALD         8872   JP1_563495872
4         GOLD         9751   JP1_563465641
5  GRANDMASTER          862   JP1_570949117
6         IRON         7608   JP1_563227866
7       MASTER         1788   JP1_570943241
8     PLATINUM         8904   JP1_563477657
9       SILVER         8972   JP1_563573099


# 3. データベースに格納

## 3-1. 関数：jsonファイルをデータベースに格納

In [3]:
def migrate_json_to_sql(json_file, db_name='lol_solorank_data.db'):

    tier = json_file.split('/')[1].split('_')[0]  # ファイル名からランクを抽出
    division = json_file.split('/')[1].split('_')[1]  # ファイル名からティアを抽出

    # データベースに接続
    conn = sqlite3.connect(db_name)
    cur = conn.cursor()


    # テーブル作成 (重複を許さない PRIMARY KEY 設定)
    cur.execute('''
        CREATE TABLE IF NOT EXISTS matches (
            match_id TEXT PRIMARY KEY,
            tier TEXT,
            division TEXT,
            data TEXT
        )
    ''')

    # JSONファイルを読み込む
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            all_data = json.load(f)
    except FileNotFoundError:
        print(f"エラー: {json_file} が見つかりません。")
        return
    print(f"{len(all_data)}件のデータを移行しています...")

    # データを1件ずつ流し込む
    count = 0
    for match in all_data:
        # 試合IDを抽出 (Riotの構造: metadata -> matchId)
        m_id = match.get('metadata', {}).get('matchId')
      
        if m_id:
            # INSERT OR IGNORE を使えば、重複していてもエラーにならず飛ばしてくれる
            cur.execute(
                "INSERT OR IGNORE INTO matches (match_id, tier, division, data) VALUES (?, ?, ?, ?)",
                (m_id, tier, division, json.dumps(match, ensure_ascii=False))
            )
            count += 1

    # 確定して終了
    conn.commit()
    conn.close()
    print(f"成功！ {count}件のデータを '{db_name}' に集約しました。")


### 3-1-1. テーブルリセット

In [ ]:
# # データベースに接続
# conn = sqlite3.connect('lol_data.db')
# cur = conn.cursor()
# 今ある古い（構造が違う）テーブルを消去する
# cur.execute("DROP TABLE IF EXISTS matches")

# # テーブル作成 (重複を許さない PRIMARY KEY 設定)
# cur.execute('''
#     CREATE TABLE IF NOT EXISTS matches (
#         match_id TEXT PRIMARY KEY,
#         tier TEXT,
#         division TEXT,
#         data TEXT
#     )
# ''')
# # 確定して終了
# conn.commit()
# conn.close()

## 3-2. 関数：jsonファイルからデータベースに格納の実行

In [4]:
# ランク・ティアのリスト初期化
Get_tier_list = ['IRON', 'BRONZE', 'SILVER', 'GOLD', 'PLATINUM', 'EMERALD', 'DIAMOND']
Get_division_list = ['IV', 'III', 'II', 'I']

In [ ]:
folder_path = 'E:\match_data_json'  # JSONファイルが入っているフォルダ名
files = os.listdir(folder_path) # フォルダ内の全ファイル名を取得

for file_name in files:
    print(f"現在処理中: {file_name}")
    parts = file_name.split('_')
    tier = parts[0]
    division = parts[1]
    file_full_path = os.path.join(folder_path, file_name)
    print(f"ファイルパス: {file_full_path}")

    # データベースに接続
    conn = sqlite3.connect('lol_data.db')
    cur = conn.cursor()

    # テーブル作成 (重複を許さない PRIMARY KEY 設定)
    cur.execute('''
        CREATE TABLE IF NOT EXISTS matches (
            match_id TEXT PRIMARY KEY,
            tier TEXT,
            division TEXT,
            data TEXT
        )
    ''')

    # JSONファイルを読み込む
    try:
        with open(file_full_path, 'r', encoding='utf-8') as f:
            all_data = json.load(f)
    except FileNotFoundError:
        print(f"エラー: {file_full_path} が見つかりません。")
        continue
    print(f"{len(all_data)}件のデータを移行しています...")

    # データを1件ずつ流し込む
    count = 0
    for match in all_data:
        # 試合IDを抽出 (Riotの構造: metadata -> matchId)
        m_id = match.get('metadata', {}).get('matchId')
      
        if m_id:
            # INSERT OR IGNORE を使えば、重複していてもエラーにならず飛ばしてくれる
            cur.execute(
                "INSERT OR IGNORE INTO matches (match_id, tier, division, data) VALUES (?, ?, ?, ?)",
                (m_id, tier, division, json.dumps(match, ensure_ascii=False))
            )
            count += 1

    # 確定して終了
    conn.commit()
    conn.close()
    print(f"{count}件のデータを 'lol_data.db' に集約しました。")

現在処理中: BRONZE_III_matches.json
ファイルパス: E:\match_data_json\BRONZE_III_matches.json
2500件のデータを移行しています...
成功！ 2500件のデータを 'lol_data.db' に集約しました。
現在処理中: BRONZE_II_matches.json
ファイルパス: E:\match_data_json\BRONZE_II_matches.json
2515件のデータを移行しています...
成功！ 2515件のデータを 'lol_data.db' に集約しました。
現在処理中: BRONZE_IV_matches.json
ファイルパス: E:\match_data_json\BRONZE_IV_matches.json
2522件のデータを移行しています...
成功！ 2522件のデータを 'lol_data.db' に集約しました。
現在処理中: BRONZE_I_matches.json
ファイルパス: E:\match_data_json\BRONZE_I_matches.json
2500件のデータを移行しています...
成功！ 2500件のデータを 'lol_data.db' に集約しました。
現在処理中: DIAMOND_III_matches.json
ファイルパス: E:\match_data_json\DIAMOND_III_matches.json
2506件のデータを移行しています...
成功！ 2506件のデータを 'lol_data.db' に集約しました。
現在処理中: DIAMOND_II_matches.json
ファイルパス: E:\match_data_json\DIAMOND_II_matches.json
2512件のデータを移行しています...
成功！ 2512件のデータを 'lol_data.db' に集約しました。
現在処理中: DIAMOND_IV_matches.json
ファイルパス: E:\match_data_json\DIAMOND_IV_matches.json
2524件のデータを移行しています...
成功！ 2524件のデータを 'lol_data.db' に集約しました。
現在処理中: DIAMOND_I_ma

## 3-3. SQLデータ確認

### 3-3-1. テーブル確認

In [7]:
conn = sqlite3.connect('lol_data.db')
cur = conn.cursor()

# matchesテーブルの列情報を取得
cur.execute("PRAGMA table_info(matches)")

columns = cur.fetchall()

print(f"{'ID':<3} | {'列名':<12} | {'型':<8} | {'PK'}")
print("-" * 35)
for col in columns:
    # col[0]: ID, col[1]: 名前, col[2]: 型, col[5]: プライマリキーフラグ
    print(f"{col[0]:<3} | {col[1]:<12} | {col[2]:<8} | {col[5]}")

conn.close()

ID  | 列名           | 型        | PK
-----------------------------------
0   | match_id     | TEXT     | 1
1   | tier         | TEXT     | 0
2   | division     | TEXT     | 0
3   | data         | TEXT     | 0


### 3-3-2. データ確認

In [ ]:
# データベースに接続
conn = sqlite3.connect('lol_data.db')
cur = conn.cursor()

try:
    # matchesテーブルから「最初の1行」だけを取得するSQLを実行
    cur.execute("SELECT match_id, tier, division, data FROM matches LIMIT 1")
    
    # 結果を1つだけ取り出す
    row = cur.fetchone()

    if row:
        m_id, tier, div, raw_data = row
        
        # 文字列として保存されているJSONデータをPythonの辞書形式に戻す
        match_json = json.loads(raw_data)
        
        print("--- 取得結果 ---")
        print(f"試合ID: {m_id}")
        print(f"ランク: {tier} {div}")
        
        # 試合内容が膨大なので、分かりやすい情報だけ抜粋して表示
        duration = match_json['info']['gameDuration']
        mode = match_json['info']['gameMode']
        print(f"試合モード: {mode}")
        print(f"試合時間: {duration // 60}分 {duration % 60}秒")
        
    else:
        print("エラー: データベースにデータが入っていないようです。")

except sqlite3.OperationalError as e:
    print(f"エラーが発生しました（テーブルがない可能性があります）: {e}")

finally:
    # 忘れずに接続を閉じる
    conn.close()

--- 取得結果 ---
試合ID: JP1_562993525
ランク: BRONZE III
試合モード: CLASSIC
試合時間: 26分 30秒
